# 08 · AI Interview Primer — Interactive Drill

> **Source notes:** `AI_Interview_Primer.md`

An active recall practice tool based on the full interview primer:
- **Flashcard bank** — 13 Q&A cards across 8 topics
- **Random card drill** — draw and self-score
- **LLM-graded answers** — submit your answer, get AI feedback on gaps
- **Key distinction explainer** — X vs Y breakdowns
- **Mock question generator** — novel practice questions by topic

Uses **Ollama** locally. No API key needed.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','ollama','-q'],check=True)
print('Packages ready.')
import ollama, random, textwrap
MODEL = 'phi3:mini'
def llm(system, user, temperature=0.3, max_tokens=300):
    r = ollama.chat(model=MODEL,messages=[{'role':'system','content':system},{'role':'user','content':user}],options={'temperature':temperature,'num_predict':max_tokens})
    return r['message']['content'].strip()
print('Ready. Make sure Ollama is running.')

## 1 · Flashcard Bank

In [ ]:
FLASHCARDS = [
    {'topic':'CoT & Reasoning','q':'What is Chain-of-Thought (CoT) prompting and why does it help?',
     'a':'Instructing the model to produce intermediate reasoning steps before the final answer. Improves accuracy on multi-step problems by decomposing them into verifiable sub-steps. Two forms: visible CoT (steps in output) and hidden reasoning tokens (internal scratchpad).'},
    {'topic':'CoT & Reasoning','q':'CoT vs Self-Consistency -- when do you use each?',
     'a':'CoT (single path): fast and cheap -- use for most tasks. Self-Consistency: sample N CoT paths, take majority vote -- use for high-stakes queries (medical, financial) where 5-20x token cost is acceptable.'},
    {'topic':'CoT & Reasoning','q':'PRM vs ORM -- what is the difference?',
     'a':'ORM rewards the final answer only -- allows correct answers via flawed reasoning. PRM rewards each individual reasoning step -- forces correct intermediate logic. PRM generalises better on novel problems. Used in o1-class training.'},
    {'topic':'CoT & Reasoning','q':'What is unfaithful reasoning in CoT agents?',
     'a':'When the visible chain of thought does not causally determine the final answer -- the answer is pre-decided and the chain is post-hoc rationalisation. Dangerous because it looks correct. Mitigated by tool-verified intermediate values.'},
    {'topic':'ReAct & Agents','q':'What is ReAct and what problem does it solve over plain CoT?',
     'a':'ReAct interleaves CoT reasoning with real tool calls (Thought-Action-Observation loop). CoT alone hallucinates external facts. ReAct grounds reasoning in real tool outputs. +34% on ALFWorld vs imitation learning (Yao et al., ICLR 2023).'},
    {'topic':'ReAct & Agents','q':'Name five key failure modes of ReAct agents.',
     'a':'1. Infinite tool loops. 2. Hallucinated tool outputs. 3. Context length collapse. 4. Cascade tool failures. 5. Action space confusion (wrong tool selected due to similar descriptions).'},
    {'topic':'RAG & Embeddings','q':'What are the four core RAGAS metrics?',
     'a':'Faithfulness: claims supported by context / total claims. Answer Relevance: does answer address question (embedding sim). Context Precision: relevant chunks / k. Context Recall: needed chunks retrieved / total needed.'},
    {'topic':'RAG & Embeddings','q':'Low faithfulness + high context precision -- what does this diagnose?',
     'a':'LLM is hallucinating despite good retrieval. The retriever found relevant chunks but the LLM ignored them. Fix: stricter grounding prompt, NLI claim verification.'},
    {'topic':'Vector DBs','q':'HNSW vs IVF-Flat vs Flat -- when do you use each?',
     'a':'Flat: exact search, use for <100k vectors. IVF-Flat: clustered approximate, 100k-10M vectors, recall >=95%. HNSW: graph-based, fastest query at scale, use for >1M vectors with latency requirements.'},
    {'topic':'Safety & Hallucination','q':'Name four hallucination types and their root causes.',
     'a':'Factual hallucination: distribution mismatch on rare queries. Confabulation: pattern completion without grounding. Attribution error: retrieval confusion, correct fact wrong source. Sycophantic: RLHF approval-seeking overrides accuracy.'},
    {'topic':'LLM Fundamentals','q':'Three training stages of a modern LLM assistant?',
     'a':'Stage 1 Pretraining: next-token prediction on internet scale -- learns language and world knowledge. Stage 2 SFT: fine-tuned on (instruction, good response) -- follows instructions. Stage 3 RLHF/DPO: aligned with human preferences -- helpful, harmless, honest.'},
    {'topic':'Cost & Latency','q':'What is prefix caching and how does it reduce cost?',
     'a':'KV matrices of a repeated prefix (system prompt + static few-shot) are cached. Subsequent calls reuse it -- 80-90% discounted. To use: put static content first, keep it identical across calls. Typical savings 50-80% on chat input costs.'},
    {'topic':'Fine-Tuning','q':'Explain LoRA -- intuition, math, when to use.',
     'a':'Task adaptation lives in a low-rank subspace of weight updates. LoRA freezes W and trains A (r x k) and B (d x r) where r << min(d,k). Effective update is BA. Trains r(d+k) params (~1% of full at r=16). Use for style/behaviour adaptation or distillation.'},
]
print(f'Flashcard bank: {len(FLASHCARDS)} cards across {len(set(c["topic"] for c in FLASHCARDS))} topics')
for t in sorted(set(c["topic"] for c in FLASHCARDS)): print(f'  {t}: {sum(1 for c in FLASHCARDS if c["topic"]==t)}')

## 2 · Random Flashcard Drill

Set `TOPIC_FILTER` to focus on one area, or leave as `None` for all topics.

In [ ]:
TOPIC_FILTER = None  # e.g. 'RAG & Embeddings'
pool = [c for c in FLASHCARDS if TOPIC_FILTER is None or c['topic']==TOPIC_FILTER]
card = random.choice(pool)
print(f'TOPIC:    {card["topic"]}')
print(f'\nQUESTION: {card["q"]}')
print('\nTake 30 seconds, then run the next cell to reveal the reference answer.')

In [ ]:
print('REFERENCE ANSWER:\n')
print(textwrap.fill(card['a'], 80))

## 3 · LLM-Graded Answer

Set `QUESTION` to any card question, write your answer in `MY_ANSWER`, and get AI feedback.

In [ ]:
QUESTION  = 'What is Chain-of-Thought (CoT) prompting and why does it help?'
REFERENCE = next(c['a'] for c in FLASHCARDS if c['q']==QUESTION)
MY_ANSWER = '''
CoT prompting tells the model to show its work step by step before answering.
This helps on hard reasoning tasks because the model does not skip steps.
'''.strip()

GRADE_SYS = 'You are a technical interview coach. Compare candidate answer to reference. Reply: STRENGTHS: (what candidate got right) MISSING: (absent key concepts) SCORE: X/5. Max 6 sentences.'
feedback  = llm(GRADE_SYS, f'Q: {QUESTION}\n\nReference: {REFERENCE}\n\nCandidate: {MY_ANSWER}', temperature=0.2, max_tokens=250)
print(f'Question: {QUESTION}\nYour answer: {MY_ANSWER}\n')
print('-- Feedback --'); print(feedback)

## 4 · Key Distinction Explainer

In [ ]:
PAIRS = [('PRM','ORM'),('CoT','Self-Consistency'),('RAG','Fine-tuning'),('HNSW','IVF-Flat'),('Semantic search','BM25 lexical search')]
PAIR  = PAIRS[0]  # change index
SYS   = 'Senior AI engineer. Explain the distinction: (1) what each is, (2) key difference, (3) when to use each. Max 5 sentences.'
exp   = llm(SYS, f'Concept A: {PAIR[0]}  Concept B: {PAIR[1]}', temperature=0.1, max_tokens=200)
print(f'{PAIR[0]} vs {PAIR[1]}\n'); print(textwrap.fill(exp, 80))

## 5 · Mock Question Generator

In [ ]:
TOPIC = 'RAG & Embeddings'
LEVEL = 'senior engineer'
N     = 3
SYS   = f'You are an AI interviewer. Generate {N} distinct technical questions about "{TOPIC}" for a {LEVEL}. Test deep understanding, not definitions. Format: Q1: ... Q2: ... Q3: ...'
qs    = llm(SYS, f'Topic: {TOPIC}. Level: {LEVEL}.', temperature=0.7, max_tokens=400)
print(f'Mock Questions -- {TOPIC} ({LEVEL})\n'); print(qs)

## Study Checklist

Aim for >=4/5 on the LLM grader for each topic before the interview.

| Topic | Cards | Key distinctions |
|---|---|---|
| CoT & Reasoning | 4 | CoT vs SC, PRM vs ORM, unfaithful reasoning |
| ReAct & Agents | 2 | ReAct vs CoT, 5 failure modes |
| RAG & Embeddings | 2 | RAGAS 4 metrics, diagnostic matrix |
| Vector DBs | 1 | HNSW vs IVF vs Flat |
| Safety | 1 | 4 hallucination types |
| LLM Fundamentals | 1 | 3 training stages |
| Cost & Latency | 1 | prefix caching |
| Fine-Tuning | 1 | LoRA math, when to use |

Full primer: [AI_Interview_Primer.md](./AI_Interview_Primer.md)